# 06 — Recurrent Networks (LSTM / GRU) on Images

This notebook treats each image as a **sequence of pixel columns (or rows)** and classifies it with a recurrent network, as suggested in the course guidelines ("an image can be considered as a sequence of columns of pixels"). There is no Moodle notebook for this case, so the model is built by adapting the RNN sentiment-analysis and MNIST examples.

Beating CNNs is not the purpose of this experiment; the comparison must be interpreted from the results actually obtained. Its purpose is to understand how recurrent models process sequential inputs, the role of gates in LSTM and GRU, the effect of sequence direction and length, and why inductive bias matters when choosing an architecture.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Requisito confirmado por el correo del profesorado

RNN + atención forma parte del núcleo, junto con LSTM y GRU. Comparar un mismo encoder recurrente con pooling de estados frente a una agregación aprendida con atención; conservar datos, dimensiones y presupuesto y publicar el cambio de parámetros. Mostrar pesos de atención sobre posiciones y explicar sus límites como interpretación. Las variantes bidireccionales, capas apiladas y CNN + RNN siguen siendo ampliaciones opcionales. Este bloque está planificado, todavía sin resultados.

## Vía A · Recurrencia, LSTM/GRU y atención como extensión

Interpretamos «LSTN» del apunte como **LSTM**. LSTM y GRU son unidades recurrentes: procesan una secuencia mediante estados y compuertas; no llevan necesariamente una capa de atención. En este proyecto, los pasos son filas o columnas de píxeles. En NLP serían vectores de tokens.

Ruta didáctica propuesta: RNN sencilla → LSTM y GRU →, agregación con atención como parte del mínimo obligatorio. Comparar último estado o pooling con una atención sobre los estados de la secuencia. Self-attention puede añadirse para relacionar posiciones, pero no define una RNN ni sustituye automáticamente su recurrencia. Una CNN + LSTM/GRU sería otra híbrida opcional. En 08 se estudia el contraste con un Transformer que utiliza self-attention sin recurrencia en su arquitectura estándar.

El objetivo es entender memoria, orden, longitud de secuencia y coste. Un resultado inferior al de CNN no invalida el experimento. [Referencia para distinguir atención y recurrencia](https://arxiv.org/abs/1706.03762).


## Protocolo revisado antes del entrenamiento · 20/11/2026

La referencia común es `config/experimental_protocol_v1.json`. Es un diseño fijado antes de entrenar las redes, no una lista de resultados obtenidos. La versión equilibrada de 18.000 fotos se conserva como experimento exploratorio. Sus métricas describen las doce clases seleccionadas con ese muestreo; no estiman automáticamente el comportamiento sobre el catálogo original.

**Por defecto:** entropía cruzada sin pesos, sin aumentos y con normalización de train a la resolución de entrada. Pesos de clase y aumentos se activarán como ablaciones explícitas. Los pesos guardados en 02 son una opción calculada, no una instrucción de aplicarlos siempre. Cada experimento registrará resolución, preprocesamiento, IDs de productos, configuración, semilla y presupuesto.

**Métricas fijadas:** macro-F1 primaria; accuracy secundaria; precisión, recall, F1 y soporte por clase; matriz de confusión en recuentos y normalizada por filas. Comparar parámetros entrenables, tiempo de entrenamiento, latencia por imagen, memoria pico y hardware. Selección por macro-F1 de validation; early stopping por pérdida de validation; tres semillas (42, 123 y 2026). El test permanece cerrado hasta congelar decisiones; intervalos finales mediante remuestreo de grupos de producto, no imágenes independientes. Los baselines deterministas no necesitan simular tres entrenamientos diferentes.

### H6 · Representación secuencial fijada y reproducible

Referencia: imagen RGB64, tensor C×H×W = 3×64×64. Se recorre de izquierda a derecha: 64 columnas, cada paso con 192 características (filas y canales). No se aplana toda la imagen como una secuencia de 12.288 escalares. LSTM/GRU procesan así una longitud razonable para el presupuesto local.

Ablación principal: filas, de arriba abajo, también 64 pasos de 192 características; cambiar solo la orientación y mantener tamaño oculto, capas, regla de parada y presupuesto. Una variante por parches de 8×8 produce 64 pasos con 192 valores por parche a esta resolución, pero cambia el vecindario representado: documentar orden raster y aplanado. Un encoder aprendido de parches añadiría parámetros y ya no sería el mismo contraste. Los píxeles se normalizan con train antes de construir la secuencia.

Registrar forma de entrada, orden, longitud, dimensión, parámetros, tiempo y memoria. Comparar con CNN/Dense RGB64 sobre los mismos IDs. La conclusión separará el efecto de recurrencia del de la representación; no se escogerá una secuencia deliberadamente desfavorable para RNN.


## Objectives

- Convert each image into a sequence (columns or rows; grayscale or RGB as features per step).
- Train a simple RNN, an LSTM and a GRU classifier using the final hidden state (or pooled states).
- Compare unidirectional and bidirectional variants and stacked layers.
- Study the effect of image resolution (sequence length) on accuracy, training time and gradient problems.
- Optionally combine a small CNN encoder with an LSTM over feature-map columns (CNN + RNN).
- Compare the recurrent models with the Dense and CNN results.

## Preview of the work

1. **Images as sequences.** reshaping, sequence length and feature dimension; visual example.
2. **Vanilla RNN baseline.** training behaviour and vanishing gradients.
3. **LSTM and GRU.** gates, number of parameters, learning curves.
4. **Bidirectional and stacked variants.** columns vs. rows reading order.
5. **Optional CNN + LSTM hybrid.** convolutional features read as a sequence.
6. **Validation results and conclusions.** what recurrence adds or misses for images.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders from notebook 02 (low-resolution variant) |
| **Outputs** | Metrics, histories and checkpoints in `results/` |
| **Course material** | `PRDL_MLLB_RecurrentNetworks.pptx`, `MLLB_Embeddings_RNN_SentimentAnalysis.ipynb`, `TO_DO_MLLB_Embeddings_TwoLayers_RNN_SentimentAnalysis.ipynb`, guideline slide "Image: RNN (LSTM, GRU…)" |